# Projet Intégrateur — Partie 1 : RAG Agentique Multi Modal

**MaintiDoc** : assistant documentaire pour techniciens de maintenance. Il exploite des documents (texte, tableaux, images) pour répondre à des questions techniques **avec des références vérifiables**.

Étapes de la partie 1 :
1. Sélection du document (PDF avec texte, images et tableaux : *Attention is all you need*)
2. Extraction et indexation : LlamaIndex, embeddings open source, PGVector (PostgreSQL)
3. Traitement des images : CLIP + descriptions textuelles indexées
4. Question-réponse : pipeline RAG avec un LLM (Azure OpenAI `gpt-4.1-mini` ; Qwen via Ollama en option)
5. Interface Streamlit (`app.py`)
6. Orchestration agentique avec LangChain

Le code réutilisable est dans le package `maintidoc/` (aussi utilisé par l'interface Streamlit). Ce notebook déroule et explique chaque étape.

## Étape 0 — Préparation

Avant d'exécuter ce notebook :
- fichier `.env` rempli (clés Azure OpenAI, voir `.env.example`)
- `docker compose up -d` : démarre PostgreSQL + pgvector (port 5433)
- `uv sync`, puis choisir le noyau `.venv` du projet dans VS Code

**Modèles utilisés**
| Rôle | Modèle |
|---|---|
| LLM (génération, agent) | Azure OpenAI `gpt-4.1-mini` (ou Qwen 2.5 via Ollama avec `LLM_PROVIDER=ollama`) |
| Description des images | le même modèle (`gpt-4.1-mini` accepte les images) |
| Embeddings texte | `intfloat/multilingual-e5-small`, open source, calculé en local |
| Embeddings images | CLIP `clip-ViT-B-32` + encodeur texte multilingue compatible |

La cellule suivante vérifie que PostgreSQL et le LLM répondent.

In [ ]:
import sqlalchemy
from IPython.display import Markdown, Image as IPImage, display

from maintidoc import config
from maintidoc.indexation import url_postgres
from maintidoc.llm import chat_model, nom_modele

with sqlalchemy.create_engine(url_postgres()).connect() as connexion:
    print("PostgreSQL :", connexion.execute(sqlalchemy.text("select version()")).scalar()[:40])
    connexion.execute(sqlalchemy.text("CREATE EXTENSION IF NOT EXISTS vector"))
    connexion.commit()
    print("pgvector   :", connexion.execute(sqlalchemy.text("select extversion from pg_extension where extname='vector'")).scalar())

print("LLM        :", nom_modele())
print(chat_model().invoke("Bonjour, présente-toi en une phrase.").content)

## Étape 1 — Sélection du document

Comme recommandé, on commence avec l'article **« Attention is all you need »** : il contient du texte, des figures (architecture du Transformer, attention multi-têtes) et des tableaux (résultats BLEU, variantes du modèle).

In [ ]:
import fitz  # PyMuPDF

pdf = config.pdf_par_defaut()
with fitz.open(pdf) as doc:
    print(pdf.name, "-", doc.page_count, "pages")
    for num, page in enumerate(doc, start=1):
        nb_images = len(page.get_images())
        nb_tableaux = len(page.find_tables().tables)
        if nb_images or nb_tableaux:
            print(f"  page {num:2d} : {nb_images} image(s), {nb_tableaux} tableau(x)")

## Étape 2 — Extraction et indexation

- **Texte** : `PyMuPDFReader` de LlamaIndex lit le PDF page par page, puis `SentenceSplitter` découpe en morceaux de 512 tokens (chevauchement 64). Chaque morceau garde son numéro de page pour les citations.
- **Tableaux** : détectés par PyMuPDF (`find_tables`), convertis en Markdown et stockés comme nœuds LlamaIndex de type `tableau`.
- **Images** : enregistrées en PNG dans `data/images/` (traitées à l'étape 3).

In [ ]:
from maintidoc.extraction import extraire_textes, extraire_tableaux, extraire_images

textes = extraire_textes(pdf)
tableaux = extraire_tableaux(pdf)
images = extraire_images(pdf, config.DOSSIER_IMAGES)

print(f"{len(textes)} morceaux de texte, {len(tableaux)} tableaux, {len(images)} images")
print("\nExemple de morceau :", textes[0].metadata)
print(textes[0].get_content()[:500])

In [ ]:
# Premier tableau détecté, en Markdown
if tableaux:
    print(tableaux[0].metadata)
    display(Markdown(tableaux[0].get_content()))

**Embeddings et stockage.** Le modèle open source `intfloat/multilingual-e5-small` (384 dimensions) est multilingue : on peut poser les questions en français sur un document en anglais. Les vecteurs et les métadonnées (type, fichier, page) sont stockés dans PostgreSQL grâce à `PGVectorStore`.

In [ ]:
from maintidoc.indexation import vider_tables, indexer_textes

vider_tables()  # repartir d'un index vide
index = indexer_textes(textes + tableaux)

In [ ]:
# Vérification directement dans PostgreSQL
with sqlalchemy.create_engine(url_postgres()).connect() as connexion:
    lignes = connexion.execute(sqlalchemy.text(
        f"select metadata_->>'type' as type, count(*) from data_{config.TABLE_TEXTE} group by 1"
    )).all()
lignes

## Étape 3 — Traitement des images

1. **Descriptions** : le modèle vision (`gpt-4.1-mini` sur Azure, ou `qwen2.5vl:3b` via Ollama) décrit chaque image. Ces descriptions sont ajoutées à l'index texte : les images deviennent interrogeables comme du texte.
2. **CLIP** : chaque image est aussi représentée par un vecteur CLIP (512 dimensions), stocké dans une seconde table PGVector. L'encodeur texte multilingue de CLIP place une question dans le même espace : on peut retrouver une image directement à partir d'une question.

In [ ]:
for img in images:
    print(f"page {img.page} : {img.chemin.name}")
    display(IPImage(filename=str(img.chemin), width=400))

In [ ]:
from maintidoc.images import decrire_image
from maintidoc.extraction import noeuds_descriptions

for img in images:
    img.description = decrire_image(img.chemin)
    print(f"--- page {img.page}\n{img.description}\n")

index.insert_nodes(noeuds_descriptions(images))  # descriptions -> index texte

In [ ]:
from maintidoc.indexation import indexer_images, rechercher_images_clip

store_images = indexer_images(images)

# Recherche texte -> image avec CLIP
for noeud, score in rechercher_images_clip(store_images, "schéma de l'architecture encodeur-décodeur", k=2):
    print(f"score {score:.3f} - page {noeud.metadata['page']}")
    display(IPImage(filename=noeud.metadata["chemin"], width=350))

## Étape 4 — Question-réponse et génération

Pipeline RAG : on récupère les 5 passages les plus proches de la question, on les place dans le prompt avec leur numéro et leur page, puis le LLM (Azure `gpt-4.1-mini`) génère une réponse fondée sur ce contexte, avec citations.

In [ ]:
from maintidoc.rag import MoteurRAG, formater_passages

moteur = MoteurRAG(index=index, store_images=store_images)

# Ce que le retriever trouve
passages = moteur.rechercher("Qu'est-ce que le multi-head attention ?", k=3)
print(formater_passages(passages, max_caracteres=300))

In [ ]:
questions = [
    "Qu'est-ce que le mécanisme de multi-head attention et pourquoi l'utiliser ?",
    "Quel score BLEU le Transformer obtient-il en traduction anglais-allemand ?",
    "Que montre la figure de l'architecture du Transformer ?",
]
for q in questions:
    reponse, sources = moteur.repondre(q)
    display(Markdown(f"**Q : {q}**\n\n{reponse}\n\n_Pages : {sorted({s.node.metadata['page'] for s in sources})}_"))

## Étape 5 — Interface utilisateur

L'interface Streamlit est dans `app.py` : chargement du PDF, indexation, saisie des questions, affichage des réponses avec leurs sources et images. Lancer dans un terminal, depuis le dossier du projet :

```
uv run streamlit run app.py
```

Elle propose deux modes : **RAG simple** (étape 4) et **Agent LangChain** (étape 6).

## Étape 6 — Orchestration agentique (LangChain)

Le RAG devient agentique : un agent LangChain (`create_agent` + `AzureChatOpenAI`) dispose de trois **outils** et décide lui-même lesquels appeler :
- `rechercher_documentation` : passages de texte et descriptions d'images
- `rechercher_tableaux` : uniquement les tableaux (filtre sur la métadonnée `type`)
- `rechercher_figures` : images retrouvées par CLIP

**Gestion du contexte** : la conversation est mémorisée par fil (`InMemorySaver` + `thread_id`), ce qui permet les questions de suivi, et les passages renvoyés par les outils sont tronqués pour ne pas saturer la fenêtre du modèle.

In [ ]:
from maintidoc.agent import construire_agent

agent = construire_agent(moteur)

def demander(question, fil="demo"):
    r = agent.demander(question, fil=fil)
    pages = sorted({s["page"] for s in r["sources"]})
    display(Markdown(f"**Q : {question}**\n\n{r['reponse']}\n\n_Outils : {r['outils']} — pages : {pages}_"))

demander("Compare les scores BLEU du Transformer base et big.")

In [ ]:
demander("Montre-moi le schéma de l'attention multi-têtes et explique-le.")

In [ ]:
# Question de suivi : l'agent se souvient de la conversation (même fil)
demander("Et combien de têtes d'attention utilise le modèle de base ?")

### Requête multimodale (texte + image)

Comme sur le schéma du cours (« Query (text, image) »), la question peut être accompagnée d'une photo. La photo est décrite par le modèle vision et comparée aux figures du document avec l'encodeur image de CLIP ; ces informations sont ajoutées à la question avant l'appel à l'agent. Ici, on utilise une figure extraite comme « photo » de test.

In [ ]:
photo = images[0].chemin
display(IPImage(filename=str(photo), width=300))
r = agent.demander("Que représente cette image et à quoi sert-elle dans le modèle ?", fil="photo", image=photo)
display(Markdown(r["reponse"]))
print("Pages des figures similaires :", [s["page"] for s in r["sources"] if s["outil"] == "photo_jointe"])

## Conclusion

La partie 1 est en place : extraction multimodale avec LlamaIndex et PyMuPDF, index vectoriel PGVector, recherche d'images par CLIP, réponses sourcées par le LLM Azure OpenAI, interface Streamlit et agent LangChain qui choisit ses outils.

Suite du projet (feuille de route du prof) : étape 2, reranking et fine-tuning d'un petit reranker multilingue (800 à 1 500 exemples, comparaison nDCG@5 / MRR@10), sorties structurées Pydantic ; étape 3, workflow LangGraph et serveur MCP (historique, tickets avec approbation humaine) ; étape 4, bot Telegram/Discord, sécurité et déploiement Docker Compose.